<a href="https://colab.research.google.com/github/Peeyusj/bpe-tokenizer/blob/main/bpe_tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
text = "hello😂"
bytes_list = text.encode("utf-8")
print(bytes_list)
print(list(bytes_list))

b'hello\xf0\x9f\x98\x82'
[104, 101, 108, 108, 111, 240, 159, 152, 130]


In [2]:
def get_stats(tokens):
    counts = {}
    for i in range(len(tokens)-1):
        pair = (tokens[i], tokens[i+1])
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# test it
text = "aaabdaaabac"
tokens = list(text.encode("utf-8"))
print(tokens)

stats = get_stats(tokens)
print(stats)

[97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
{(97, 97): 4, (97, 98): 2, (98, 100): 1, (100, 97): 1, (98, 97): 1, (97, 99): 1}


In [4]:
def merge(tokens, pair, new_token):
    new_tokens = []
    i = 0
    while i < len(tokens) - 1:
        if tokens[i] == pair[0] and tokens[i+1] == pair[1]:  # blank 1
            new_tokens.append(new_token)  # blank 2
            i += 2  # blank 3
        else:
            new_tokens.append(tokens[i])
            i += 1
    new_tokens.append(tokens[-1])
    return new_tokens

In [5]:
tokens = list("aaabdaaabac".encode("utf-8"))
print("before:", tokens)

stats = get_stats(tokens)
best_pair = max(stats, key=stats.get)
print("merging:", best_pair)

tokens = merge(tokens, best_pair, 256)
print("after:", tokens)

before: [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
merging: (97, 97)
after: [256, 97, 98, 100, 256, 97, 98, 97, 99]


In [6]:
tokens = list("aaabdaaabac".encode("utf-8"))
merges = {}
num_merges = 20

for i in range(num_merges ):  # blank 1 - how many times?
    stats = get_stats(tokens)
    best_pair = max(stats, key=stats.get)
    new_token = 256 + i  # blank 2 - starts at 256, increases by 1
    tokens = merge(tokens, best_pair, new_token)
    merges[best_pair] = new_token  # saving the rule
    print(f"merge {i+1}: {best_pair} -> {new_token}")

merge 1: (97, 97) -> 256
merge 2: (256, 97) -> 257
merge 3: (257, 98) -> 258
merge 4: (258, 100) -> 259
merge 5: (259, 258) -> 260
merge 6: (260, 97) -> 261
merge 7: (261, 99) -> 262
merge 8: (262, 99) -> 263
merge 9: (263, 99) -> 264
merge 10: (264, 99) -> 265
merge 11: (265, 99) -> 266
merge 12: (266, 99) -> 267
merge 13: (267, 99) -> 268
merge 14: (268, 99) -> 269
merge 15: (269, 99) -> 270
merge 16: (270, 99) -> 271
merge 17: (271, 99) -> 272
merge 18: (272, 99) -> 273
merge 19: (273, 99) -> 274
merge 20: (274, 99) -> 275


In [7]:
print("final tokens:", tokens)
print("total merges saved:", len(merges))

final tokens: [275, 99]
total merges saved: 20


In [9]:
print(bytes([97]))
print(bytes([104, 101, 108, 108, 111]))

b'a'
b'hello'


In [10]:
vocab = {i: bytes([i]) for i in range(256)}

for (a, b), new_token in merges.items():
    vocab[new_token] = vocab[a] + vocab[b]

print(vocab[256])
print(vocab[257])
print(vocab[258])

b'aa'
b'aaa'
b'aaab'
